# CNN 1D Univariate

In this section we implement the CNN 1D Model (1D Convolutional Neural Network) using the **TimeSeriesDataset** approach with one-hot encoding.

The CNN 1D (1D Convolutional Neural Network) Forecaster is designed for time series forecasting. It uses one-hot encoding to identify individual series (1502 unique series), processing one series at a time. It uses convolutional layers to capture local temporal patterns and dependencies efficiently, processing sequences in parallel rather than sequentially.

With this appraoch each training sample represents a single series with its one-hot encoded identifier, allowing the model to learn series-specific local patterns through convolutional filters.

## Architecture

```bash
Input (seq_length, input_size)
    ↓
Transpose (input_size, seq_length)
    ↓
Conv1D Block 1
  ├─ Conv1D (input_size → hidden_size, kernel=3)
  ├─ ReLU
  └─ MaxPool1D
    ↓
Conv1D Block 2
  ├─ Conv1D (hidden_size → hidden_size, kernel=3)
  ├─ ReLU
  └─ MaxPool1D
    ↓
Conv1D Block 3
  └─ (same structure)
    ↓
Adaptive Average Pooling (→ 1)
    ↓
Flatten
    ↓
Fully Connected (hidden_size → hidden_size)
    ↓
ReLU + Dropout
    ↓
Fully Connected (hidden_size → 1)
    ↓
Output (1 prediction)
```

## Layer Breakdown

- **Conv1D Blocks:** 3 stacked convolutional blocks (configurable)
- **Kernel Size:** 3 (captures patterns across 3 consecutive timesteps)
- **Hidden Size:** 64 filters per layer (default)
- **MaxPooling:** Applied after each convolution with stride=1, padding=1
- **Adaptive Average Pooling:** Reduces variable-length sequences to fixed size
- **Fully Connected Layers:** Two FC layers with ReLU activation between them
- **Dropout:** Applied before final output layer
- **Output Layer:** Single neuron producing 1-step forecast

## Advantages

- **Parallel Processing:** Can process entire sequence at once (faster than RNN/LSTM/GRU)
- **Fewer Parameters:** Generally requires fewer parameters than RNN-based models
- **No Vanishing Gradients:** No recurrent connections means no vanishing gradient problem

## Limitations

- **Limited Long-Range Dependencies:** Receptive field grows slowly with depth; may miss long-term patterns
- **Fixed Kernel Size:** Kernel size (3) determines the temporal context window
- **Less Interpretable:** Harder to understand what patterns each filter captures
- **Sequence Length Sensitivity:** Very short sequences may not benefit from multiple conv layers

## When to Use

- Need fast training and inference
- Local temporal patterns are important (e.g., sudden spikes, dips)
- Dataset is moderate to large (>5,000 samples)
- Sequences are short to medium length (5-50 timesteps)
- Parallel processing capability is valuable

## Key Hyperparameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| hidden_size | 64 | Number of convolutional filters - more filters capture more patterns |
| num_layers | 3 | Number of convolutional blocks - depth increases receptive field |
| kernel_size | 3 | Size of convolution window - fixed at 3 timesteps |
| dropout | 0.2 | Dropout rate for regularization |



## Model

In [ ]:
import torch 
import torch.nn as nn

In [ ]:
class CNN1DForecaster(nn.Module):
    """
    1D CNN model for MULTIVARIATE time series forecasting.
    Architecture: 
        Conv1D blocks (Conv -> ReLU -> MaxPool) -> 
        Adaptive Average Pooling -> Flatten -> 
        Fully Connected -> Dropout -> Output
    
    CNNs can capture local patterns and temporal dependencies efficiently.
    Uses multiple kernel sizes to capture patterns at different scales.
    Processes sequences in parallel (unlike RNN/LSTM/GRU).
    """
    def __init__(self, input_size, hidden_size=64, num_layers=3, dropout=0.2):
        """
        Args:
            input_size: Number of input features (Value + year + month + one-hot)
            hidden_size: Number of filters in conv layers
            num_layers: Number of convolutional blocks (minimum 1)
            dropout: Dropout rate
        """
        super(CNN1DForecaster, self).__init__()
        
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = max(1, num_layers)
        
        # Convolutional layers
        conv_layers = []
        
        # First conv block: input_size -> hidden_size
        conv_layers.extend([
            nn.Conv1d(in_channels=input_size, out_channels=hidden_size, 
                     kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=1, padding=1)
        ])
        
        # Additional conv blocks: hidden_size -> hidden_size
        for i in range(1, self.num_layers):
            conv_layers.extend([
                nn.Conv1d(in_channels=hidden_size, out_channels=hidden_size, 
                         kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool1d(kernel_size=2, stride=1, padding=1)
            ])
        
        self.conv_blocks = nn.Sequential(*conv_layers)
        
        # Adaptive pooling to fixed size output
        self.adaptive_pool = nn.AdaptiveAvgPool1d(1)
        
        # Fully connected layers
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        
        # Conv1d expects (batch_size, channels, seq_length)
        # Transpose from (batch, seq, features) to (batch, features, seq)
        x = x.transpose(1, 2)  # (batch_size, input_size, seq_length)
        
        # Apply convolutional blocks
        x = self.conv_blocks(x)  # (batch_size, hidden_size, seq_length')
        
        # Adaptive pooling to reduce to (batch_size, hidden_size, 1)
        x = self.adaptive_pool(x)  # (batch_size, hidden_size, 1)
        
        # Flatten
        x = x.squeeze(-1)  # (batch_size, hidden_size)
        
        # Fully connected layers
        x = self.fc1(x)  # (batch_size, hidden_size)
        x = self.relu(x)
        x = self.dropout(x)
        out = self.fc2(x)  # (batch_size, 1)
        
        return out


### Model Results without Exogenous Features

In this section, the CNN 1D model is evaluated based on temporal features (value, year, and month) and one-hot encoded series identifiers, without the incorporation of external economic indicators. This baseline approach enables assessment of how well the model captures local temporal patterns using only historical information and series-specific context through convolutional filters. A 3-fold time series cross-validation strategy is employed to ensure robust performance evaluation and prevent data leakage.

### Optuna Hyperparameter Search Results

| Trial | Validation Loss | Batch Size | Dropout | Hidden Size | Learning Rate | Num Layers | Duration |
|------:|----------------:|----------:|--------:|-----------:|--------------:|----------:|---------:|
| 0 | 0.30484 | 64 | 0.11543 | 128 | 0.00228 | 3 | 15.48s |
| 1 | 0.31083 | 64 | 0.10251 | 128 | 0.00063 | 3 | 43.26s |
| 2 | 0.30445 | 32 | 0.19464 | 128 | 0.00320 | 4 | 27.75s |





###Best Hyperparameters

Validation Loss: 0.30445

Parameters:
  - learning_rate: 0.00320
  - batch_size: 32
  - num_layers: 4
  - hidden_size: 128
  - dropout: 0.19464

#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/univariate/cnn_1d/fold1/fold_results.png)

#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 2 Results](./img/univariate/cnn_1d/fold2/fold_results.png)

#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30

![Fold 3 Results](./img/univariate/cnn_1d/fold3/fold_results.png)

### Fold Results
| Fold | MSE | RMSE | MAE | R² | SMAPE |
|------|----------|----------|----------|------|-------|
| Fold 1 | 108887.48 | 329.98 | 135.31 | 0.8048 | 64.96% |
| Fold 2 | 109589.15 | 331.04 | 136.54 | 0.7814 | 80.17% |
| Fold 3 | 53795.70 | 231.94 | 98.17 | 0.8976 | 72.64% |
| **Average** | **90757.44 ± 32127.70** | **297.65 ± 54.04** | **123.34 ± 20.41** | **0.8279 ± 0.0598** | **72.60% ± 7.70%** |

### Average SMAPE Distribution Across Folds

| SMAPE Range | Percentage of Series | Number of Series (avg) |
|-------------|----------------------|------------------------|
| <10% | 4.6% ± 3.5% | 61 |
| 10-20% | 10.2% ± 5.4% | 137 |
| 20-30% | 12.9% ± 2.3% | 172 |
| 30-40% | 12.3% ± 1.0% | 164 |
| >40% | 60.0% ± 8.2% | 800 |




**Comparison with Baseline:**

The CNN 1D univariate model with one-hot encoding achieves an average SMAPE of **72.60% ± 7.70%**, which is comparable to the baseline 3-month rolling average (72.26% ± 7.06%). The CNN 1D model demonstrates competitive performance with only a **0.34 percentage point difference**, suggesting that the convolutional architecture effectively captures local temporal patterns with similar accuracy to the simple rolling average baseline.

### Model Results with Exogenous Features